# Team Workbook — Day 4
## Predictive Analytics II + Unsupervised Learning  ·  Units III & IV

Yesterday you built a model and scored it once. Today you find out whether that score was real,
make it better on purpose rather than by luck, and then look at what your data says with no target
at all.

**Three things change today:**

1. **One split lies.** You will score every model five times instead of once.
2. **Tuning beats guessing.** You will search for settings rather than pick them.
3. **Not everything has a target.** Clustering and PCA find structure without one.

---


## Step 0 — Setup


In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
sns.set_theme(style="whitegrid"); plt.rcParams["figure.figsize"] = (7, 4)

FILE     = "day2_clean.csv"
NUM_COL  = "____"
CAT_COL  = "____"
TARGET   = None
TASK     = "auto"

df = pd.read_csv(FILE); df.columns = df.columns.str.strip()
df[NUM_COL] = pd.to_numeric(df[NUM_COL], errors="coerce")
print(df.shape)


### Paste the modelling kit again


In [ ]:
"""
MLKIT  —  the modelling helpers used on Days 3, 4 and 5.
Written so that the SAME code works on every team's dataset.
Paste this whole cell once, near the top of your notebook.
"""
import numpy as np, pandas as pd, warnings
warnings.filterwarnings("ignore")
from sklearn.model_selection import train_test_split, cross_val_score, KFold, StratifiedKFold
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score,
                             accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, roc_auc_score)

MAX_ONEHOT = 25          # a category with more levels than this is dropped
MAX_NULL_FRAC = 0.5      # a column emptier than this is dropped


# ---------------------------------------------------------------- target
def make_target(df, num_col=None, target=None, task="auto", verbose=True):
    """
    Returns (y, task, derived_from, label).
    target=None  -> derive a yes/no target by splitting num_col at its median.
                    This works on ANY dataset that has one number in it.
    """
    derived_from = None
    if target is None:
        if num_col is None:
            raise ValueError("Give either target= or num_col= so a target can be derived.")
        s = pd.to_numeric(df[num_col], errors="coerce")
        cut = s.median()
        y = (s >= cut).astype("float")
        y[s.isna()] = np.nan
        task, derived_from = "classification", num_col
        label = f"is {num_col} at or above its median ({cut:,.2f})?"
    else:
        s = df[target]
        if pd.api.types.is_numeric_dtype(s):
            nunique = s.nunique(dropna=True)
            if task == "auto":
                task = "classification" if nunique <= 10 else "regression"
            y = pd.to_numeric(s, errors="coerce")
        else:
            task = "classification" if task == "auto" else task
            levels = sorted(s.dropna().astype(str).unique())
            if len(levels) == 2:
                mapping = {levels[0]: 0, levels[1]: 1}
                y = s.astype(str).map(mapping)
                y[s.isna()] = np.nan
                print(f"ENCODED  : {levels[0]} -> 0    {levels[1]} -> 1  "
                      f"(so \"positive\" below means {levels[1]})")
            else:
                y = s.astype(str)
        label = f"{target}"

    if verbose:
        print(f"TARGET   : {label}")
        print(f"TASK     : {task}")
        if derived_from:
            print(f"           (derived from {derived_from} - so {derived_from} MUST be excluded from the features)")
        if task == "classification":
            vc = pd.Series(y).value_counts(dropna=True)
            share = (vc / vc.sum() * 100).round(1)
            print(f"BALANCE  : " + "  ".join(f"{k}={v}%" for k, v in share.items()))
            if share.max() > 80:
                print("           WARNING - one class dominates. Accuracy will look great and mean nothing.")
    return y, task, derived_from, label


# ---------------------------------------------------------------- features
def make_features(df, y, derived_from=None, target=None, drop=None, verbose=True):
    """One-hot encodes, imputes, and removes anything that would leak or be useless."""
    drop = set(drop or [])
    if derived_from: drop.add(derived_from)
    if target:       drop.add(target)

    X = df.copy()
    n = len(X)
    removed = []

    for c in list(X.columns):
        if c in drop:
            removed.append((c, "excluded on purpose - it would leak the answer" if c == derived_from
                               else "this is the target")); X = X.drop(columns=c); continue
        s = X[c]
        if s.isnull().mean() > MAX_NULL_FRAC:
            removed.append((c, f"{s.isnull().mean()*100:.0f}% empty")); X = X.drop(columns=c); continue
        if s.nunique(dropna=True) <= 1:
            removed.append((c, "same value in every row")); X = X.drop(columns=c); continue
        if pd.api.types.is_datetime64_any_dtype(s):
            removed.append((c, "a date - engineer month/weekday from it first")); X = X.drop(columns=c); continue
        if not pd.api.types.is_numeric_dtype(s):
            u = s.nunique(dropna=True)
            if u == n:
                looks_dated = pd.to_datetime(s, errors="coerce", format="mixed").notna().mean() > 0.8
                removed.append((c, "a date - engineer month/weekday from it first" if looks_dated
                                   else "one value per row - an ID, not a feature"))
                X = X.drop(columns=c); continue
            if u > MAX_ONEHOT:
                removed.append((c, f"{u} categories - too many to one-hot")); X = X.drop(columns=c); continue
        else:
            if pd.api.types.is_integer_dtype(s) and s.nunique(dropna=True) == n and n > 10:
                removed.append((c, "one value per row - an ID, not a feature")); X = X.drop(columns=c); continue

    num_cols = X.select_dtypes(include=np.number).columns.tolist()
    cat_cols = [c for c in X.columns if c not in num_cols]

    for c in num_cols:
        if X[c].isnull().any():
            X[c] = X[c].fillna(X[c].median())
    for c in cat_cols:
        X[c] = X[c].astype(str).str.strip().fillna("missing")

    X = pd.get_dummies(X, columns=cat_cols, drop_first=True, dtype=float)

    keep = pd.Series(y).notna().values
    X, y = X.loc[keep], pd.Series(y).loc[keep]

    if verbose:
        print(f"\nFEATURES : {X.shape[1]} columns built from {len(num_cols)} numeric + {len(cat_cols)} categorical")
        print(f"ROWS     : {len(X)} usable")
        if removed:
            print("REMOVED  :")
            for c, why in removed[:12]:
                print(f"           {c:<24} {why}")
    return X, y


# ---------------------------------------------------------------- split
def split(X, y, task, test_size=0.25, seed=42):
    strat = y if (task == "classification" and pd.Series(y).value_counts().min() >= 2) else None
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=test_size, random_state=seed, stratify=strat)
    print(f"TRAIN    : {len(Xtr)} rows      TEST: {len(Xte)} rows  (the test rows are never used to fit)")
    return Xtr, Xte, ytr, yte


# ---------------------------------------------------------------- metrics
def score(y_true, y_pred, task, y_prob=None):
    if task == "regression":
        return {"MAE": mean_absolute_error(y_true, y_pred),
                "RMSE": mean_squared_error(y_true, y_pred) ** 0.5,
                "R2": r2_score(y_true, y_pred)}
    avg = "binary" if pd.Series(y_true).nunique() == 2 else "macro"
    out = {"Accuracy": accuracy_score(y_true, y_pred),
           "Precision": precision_score(y_true, y_pred, average=avg, zero_division=0),
           "Recall": recall_score(y_true, y_pred, average=avg, zero_division=0),
           "F1": f1_score(y_true, y_pred, average=avg, zero_division=0)}
    if y_prob is not None and pd.Series(y_true).nunique() == 2:
        try: out["ROC_AUC"] = roc_auc_score(y_true, y_prob)
        except Exception: pass
    return out


RESULTS = {}

def evaluate(name, model, Xtr, Xte, ytr, yte, task, store=True, verbose=True):
    """Fit, predict on the held-out test set, score, remember the result."""
    model.fit(Xtr, ytr)
    pred = model.predict(Xte)
    prob = None
    if task == "classification" and hasattr(model, "predict_proba"):
        try: prob = model.predict_proba(Xte)[:, 1]
        except Exception: pass
    s = score(yte, pred, task, prob)
    if store: RESULTS[name] = s
    if verbose:
        print(f"{name:<22} " + "   ".join(f"{k} {v:.3f}" for k, v in s.items()))
    return model, s


def baseline(Xtr, Xte, ytr, yte, task):
    """The score to beat. A model that cannot beat this has learnt nothing."""
    m = DummyRegressor(strategy="mean") if task == "regression" else DummyClassifier(strategy="most_frequent")
    _, s = evaluate("BASELINE (no model)", m, Xtr, Xte, ytr, yte, task)
    return s


def leaderboard():
    if not RESULTS:
        print("Nothing evaluated yet."); return None
    t = pd.DataFrame(RESULTS).T.round(3)
    key = "R2" if "R2" in t.columns else "F1"
    return t.sort_values(key, ascending=False)


def cv(model, X, y, task, folds=5):
    scoring = "r2" if task == "regression" else "f1_macro"
    kf = KFold(folds, shuffle=True, random_state=42) if task == "regression" \
         else StratifiedKFold(folds, shuffle=True, random_state=42)
    s = cross_val_score(model, X, y, cv=kf, scoring=scoring)
    print(f"{folds}-fold {scoring}: " + " ".join(f"{v:.3f}" for v in s))
    print(f"  mean {s.mean():.3f}   spread +/- {s.std():.3f}")
    return s


def leak_check(X, y, task, thresh=0.95):
    """
    Flags any single feature that predicts the target almost perfectly on its own.
    The automatic guard only removes the column the target was DERIVED from.
    It cannot know that m1..m5 add up to total - only you can. This finds those.
    """
    yy = pd.Series(y).astype(float) if task == "regression" else pd.Series(y)
    suspects = []
    for c in X.columns:
        col = X[c]
        if col.nunique() <= 1:
            continue
        if task == "regression":
            r = abs(np.corrcoef(col, yy)[0, 1])
        else:
            try:    r = abs(np.corrcoef(col, pd.factorize(yy)[0])[0, 1])
            except Exception: continue
        if np.isfinite(r) and r >= thresh:
            suspects.append((c, round(float(r), 3)))
    if suspects:
        print("POSSIBLE LEAK - these features almost ARE the target:")
        for c, r in sorted(suspects, key=lambda t: -t[1]):
            print(f"   {c:<28} correlation {r}")
        print("   If a feature is computed FROM the target, drop it. A model that")
        print("   sees the answer is not predicting - it is reading.")
    else:
        print("No single feature gives the answer away. (Combinations still might.)")
    return suspects


def importances(model, X, top=12):
    if hasattr(model, "feature_importances_"):
        v = pd.Series(model.feature_importances_, index=X.columns)
    elif hasattr(model, "coef_"):
        c = np.ravel(model.coef_)
        v = pd.Series(np.abs(c), index=X.columns)
    else:
        print("This model does not expose importances."); return None
    return v.sort_values(ascending=False).head(top).round(4)


In [ ]:
y, task, derived_from, label = make_target(df, num_col=NUM_COL, target=TARGET, task=TASK)
X, y = make_features(df, y, derived_from=derived_from, target=TARGET)
X_train, X_test, y_train, y_test = split(X, y, task)
base = baseline(X_train, X_test, y_train, y_test, task)


---
## Step 1 — Scaling, and why some models need it

A model that measures **distance** between rows — KNN, K-Means, anything with "neighbour" or
"cluster" in the name — is dominated by whichever column happens to have the biggest numbers.
Salary in rupees and age in years are not comparable until you put them on the same scale.

Trees and forests do not care: they split on one column at a time, so units are irrelevant.


In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler

print("Before scaling - look at the ranges:")
print(X.describe().T[["mean", "std", "min", "max"]].head(6).round(2))

scaler = StandardScaler()
X_train_s = pd.DataFrame(scaler.fit_transform(X_train), columns=X.columns, index=X_train.index)
X_test_s  = pd.DataFrame(scaler.transform(X_test),      columns=X.columns, index=X_test.index)

print("\nAfter scaling - every column now has mean 0 and std 1:")
print(X_train_s.describe().T[["mean", "std", "min", "max"]].head(6).round(2))


> **`fit_transform` on train, `transform` on test — never `fit` on the test set.** Fitting the
> scaler on all your data lets information from the test rows influence the training. That is
> leakage in a subtler form, and it is why the next cell exists.


### Step 1b — Pipelines: leakage-proof by construction


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor

KNN = KNeighborsRegressor if task == "regression" else KNeighborsClassifier
knn_pipe = Pipeline([("scale", StandardScaler()), ("model", KNN(n_neighbors=5))])

evaluate("KNN (k=5, scaled)", knn_pipe, X_train, X_test, y_train, y_test, task)
evaluate("KNN (k=5, UNSCALED)", KNN(n_neighbors=5), X_train, X_test, y_train, y_test, task)


Compare those two lines. The gap between them is what scaling is worth on your data — and it
is usually large.

A **Pipeline** bundles the scaler and the model into one object. When it is cross-validated, the
scaler is refitted inside every fold automatically, so leakage becomes impossible rather than
merely discouraged. Use pipelines from here on.


---
## Step 2 — More models

Nothing exotic. Four families cover most real problems.


In [ ]:
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, \
                             GradientBoostingClassifier, GradientBoostingRegressor

if task == "regression":
    zoo = {
        "Linear":        Pipeline([("s", StandardScaler()), ("m", LinearRegression())]),
        "Tree(d=4)":     DecisionTreeRegressor(max_depth=4, random_state=42),
        "RandomForest":  RandomForestRegressor(n_estimators=200, random_state=42),
        "GradientBoost": GradientBoostingRegressor(random_state=42),
        "KNN(k=5)":      knn_pipe,
    }
else:
    zoo = {
        "Logistic":      Pipeline([("s", StandardScaler()), ("m", LogisticRegression(max_iter=3000))]),
        "Tree(d=4)":     DecisionTreeClassifier(max_depth=4, random_state=42),
        "RandomForest":  RandomForestClassifier(n_estimators=200, random_state=42),
        "GradientBoost": GradientBoostingClassifier(random_state=42),
        "KNN(k=5)":      knn_pipe,
    }

for name, m in zoo.items():
    evaluate(name, m, X_train, X_test, y_train, y_test, task)

leaderboard()


**A random forest is many decision trees**, each grown on a random subset of the rows and columns,
voting together. Individually they are mediocre and disagree with each other; averaged, the
disagreements cancel and the signal survives. That is the whole idea, and it is why forests are
usually the strongest thing you can fit in one line.


---
## Step 3 — Cross-validation: one split lies

Your test set was 25% of your rows, chosen at random. Change `seed=42` to `seed=7` and every score
moves. So which one was true?

Cross-validation answers that by doing the split five times, using a different fifth as the test
set each time, and reporting the spread as well as the average.


In [ ]:
seeds = [0, 1, 7, 42, 99]
key = "R2" if task == "regression" else "F1"
best_name = list(zoo)[0]

vals = []
for sd in seeds:
    Xtr, Xte, ytr, yte = split(X, y, task, seed=sd)
    _, s = evaluate(f"seed={sd}", zoo[best_name], Xtr, Xte, ytr, yte, task, store=False, verbose=False)
    vals.append(s[key])

print(f"{best_name}, {key} across five different random splits:")
print("   " + "  ".join(f"{v:.3f}" for v in vals))
print(f"   lowest {min(vals):.3f}   highest {max(vals):.3f}   range {max(vals)-min(vals):.3f}")
print("\nThat range is how much your single number from yesterday could have been luck.")


In [ ]:
print("Proper 5-fold cross-validation on every model:\n")
cv_rows = {}
for name, m in zoo.items():
    s = cv(m, X, y, task, folds=5)
    cv_rows[name] = {"cv_mean": round(s.mean(), 3), "cv_std": round(s.std(), 3)}
    print()

cvt = pd.DataFrame(cv_rows).T.sort_values("cv_mean", ascending=False)
cvt["believable_range"] = (cvt["cv_mean"].round(2).astype(str) + "  +/- " + cvt["cv_std"].round(2).astype(str))
cvt


**Read the `cv_std` column before the `cv_mean` column.** If model A scores 0.72 ± 0.09 and model B
scores 0.70 ± 0.02, A is not better than B — their ranges overlap almost entirely, and B is far more
predictable. Choosing A on the strength of 0.02 is choosing noise.


---
## Step 4 — Tuning: searching instead of guessing

`max_depth=4` yesterday was a guess. `GridSearchCV` tries every combination you list, cross-validates
each one, and returns the winner. It is the same work you would do by hand, done exhaustively.


In [ ]:
from sklearn.model_selection import GridSearchCV

RF = RandomForestRegressor if task == "regression" else RandomForestClassifier
grid = {
    "n_estimators":      [100, 300],
    "max_depth":         [3, 6, 12, None],
    "min_samples_leaf":  [1, 5, 10],
}
scoring = "r2" if task == "regression" else "f1_macro"

gs = GridSearchCV(RF(random_state=42), grid, cv=5, scoring=scoring, n_jobs=-1)
gs.fit(X_train, y_train)

print(f"Combinations tried : {len(gs.cv_results_['params'])}")
print(f"Best settings      : {gs.best_params_}")
print(f"Best CV {scoring:<10} : {gs.best_score_:.3f}\n")

tuned, s_tuned = evaluate("RandomForest (tuned)", gs.best_estimator_,
                          X_train, X_test, y_train, y_test, task)


In [ ]:
res = (pd.DataFrame(gs.cv_results_)[["param_max_depth", "param_min_samples_leaf",
                                     "param_n_estimators", "mean_test_score", "std_test_score"]]
         .sort_values("mean_test_score", ascending=False).round(3))
print("Top 8 combinations:"); print(res.head(8).to_string(index=False))
print("\nWorst 3 - notice how much the settings matter:"); print(res.tail(3).to_string(index=False))


> **The tuned model is not always better on the test set**, and if yours is worse, that is worth a
> sentence in your report. Grid search optimises the cross-validation score; a small drop on one
> test set is within noise. Trust the CV number.


---
## Step 5 — Choose, and say why


In [ ]:
final = leaderboard()
print(final.to_string())
print(f"\nBaseline to beat: " + "  ".join(f"{k} {v:.3f}" for k, v in base.items()))


Pick your model on **three** grounds, not one:

| | |
|---|---|
| **Score** | does it beat the baseline by a margin bigger than the cross-validation spread? |
| **Stability** | is `cv_std` small? A model that swings wildly cannot be trusted in production. |
| **Explainability** | can you tell someone *why* it made a prediction? A tree can. A forest, less so. |

Write one sentence: *"We chose ______ because ______, even though ______ scored slightly higher."*


---
## Step 6 — Clustering: finding groups nobody labelled

Everything so far had a target. K-Means has none — it just finds rows that resemble each other.
Use it to answer *"are there natural types in my data?"*


In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

Xs = StandardScaler().fit_transform(X)      # clustering ALWAYS needs scaling

inertia, sil, ks = [], [], range(2, 9)
for k in ks:
    km = KMeans(n_clusters=k, n_init=10, random_state=42).fit(Xs)
    inertia.append(km.inertia_)
    sil.append(silhouette_score(Xs, km.labels_))

fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
ax[0].plot(list(ks), inertia, marker="o", color="#2194D4")
ax[0].set_xlabel("number of clusters (k)"); ax[0].set_ylabel("inertia")
ax[0].set_title("Elbow - look for the bend")
ax[1].plot(list(ks), sil, marker="o", color="#137A5E")
ax[1].set_xlabel("k"); ax[1].set_ylabel("silhouette")
ax[1].set_title("Silhouette - higher is better")
plt.tight_layout(); plt.show()

print("Silhouette by k:", {k: round(v, 3) for k, v in zip(ks, sil)})
print("Above 0.5 = clear clusters.  0.25-0.5 = weak.  Below 0.25 = there are no real groups.")


In [ ]:
K = 3          # <- set this from the elbow and the silhouette above

km = KMeans(n_clusters=K, n_init=10, random_state=42).fit(Xs)
df_c = df.loc[X.index].copy()
df_c["cluster"] = km.labels_

print("How big is each cluster?")
print(df_c["cluster"].value_counts().sort_index().to_string())

print(f"\nWhat is each cluster LIKE? (average {NUM_COL}, and the commonest {CAT_COL})")
profile = df_c.groupby("cluster").agg(
    n=(NUM_COL, "count"),
    avg=(NUM_COL, "mean"),
    top_category=(CAT_COL, lambda s: s.value_counts().index[0]),
).round(2)
profile


**A cluster is useless until you can name it.** Look at that profile table and give each cluster a
label a person would understand — *"high-value repeat buyers"*, *"one-off small orders"*. If you
cannot name it, either K is wrong or there are no real groups in your data. Both are reportable.


---
## Step 7 — PCA: squeezing many columns into two

You cannot plot 40 columns. PCA finds the two directions along which your data varies most, so you
can see it. It is compression, not magic — and you must report how much you threw away.


In [ ]:
from sklearn.decomposition import PCA

p_full = PCA().fit(Xs)
cum = np.cumsum(p_full.explained_variance_ratio_)
print(f"2 components keep {cum[1]*100:.1f}% of the variation")
if len(cum) > 4:  print(f"5 components keep {cum[4]*100:.1f}%")
n90 = int(np.argmax(cum >= 0.90) + 1)
print(f"You need {n90} components to keep 90%")

plt.figure(figsize=(6, 3.2))
plt.plot(range(1, len(cum)+1), cum*100, marker="o", color="#2194D4")
plt.axhline(90, color="#B4342F", ls="--", label="90%")
plt.xlabel("number of components"); plt.ylabel("% of variation kept"); plt.legend()
plt.title("How much you keep as you add components"); plt.tight_layout(); plt.show()


In [ ]:
pcs = PCA(n_components=2).fit_transform(Xs)
plt.figure(figsize=(6.4, 4.4))
sns.scatterplot(x=pcs[:, 0], y=pcs[:, 1], hue=km.labels_, palette="tab10", s=26, alpha=.8)
plt.xlabel("component 1"); plt.ylabel("component 2")
plt.title(f"Your data in 2 dimensions, coloured by cluster "
          f"({cum[1]*100:.0f}% of variation shown)")
plt.legend(title="cluster"); plt.tight_layout(); plt.show()


> **The axes mean nothing on their own.** "Component 1" is a blend of your original columns, not a
> quantity anyone can act on. PCA is for *seeing* and for *speeding models up* — never quote a
> component to a stakeholder.


---
## Step 8 — Save the model, and write its card


In [ ]:
import joblib

FINAL = tuned          # <- the model you actually chose
joblib.dump({"model": FINAL, "columns": list(X.columns), "task": task,
             "target_label": label, "derived_from": derived_from},
            "final_model.joblib")

card = f"""
MODEL CARD
==========
What it predicts   : {label}
Type               : {task}
Trained on         : {len(X_train)} rows,  tested on {len(X_test)} rows never seen in training
Features used      : {X.shape[1]}
Algorithm          : {type(FINAL).__name__}
Test performance   : """ + "  ".join(f"{k} {v:.3f}" for k, v in s_tuned.items()) + f"""
Baseline           : """ + "  ".join(f"{k} {v:.3f}" for k, v in base.items()) + f"""
Excluded to avoid
leakage            : {derived_from or 'nothing - the target was a real column'}

KNOWN LIMITS
- Trained on data from ONE source; performance elsewhere is unknown.
- Rows with more than 50% missing values were dropped, not imputed.
- Correlation only. Nothing here shows that changing a feature would change the outcome.
- <add your own: sample size, time period, groups under-represented>
"""
print(card)
open("model_card.txt", "w").write(card)


### Optional — a first look at time series  *(only if you have a date)*


In [ ]:
DATE_COL = None      # set this if you have one

if DATE_COL and DATE_COL in df.columns:
    ts = df[[DATE_COL, NUM_COL]].copy()
    ts[DATE_COL] = pd.to_datetime(ts[DATE_COL], errors="coerce", format="mixed")
    ts = ts.dropna().set_index(DATE_COL).resample("ME")[NUM_COL].mean().dropna()

    if len(ts) >= 6:
        roll = ts.rolling(3).mean()
        naive = ts.shift(1)                       # forecast = last month. The baseline.
        mae = (ts - naive).abs().mean()

        plt.figure(figsize=(9, 3.4))
        plt.plot(ts.index, ts.values, marker="o", label="actual", color="#2194D4")
        plt.plot(roll.index, roll.values, label="3-month average", color="#B4342F")
        plt.title(f"Monthly {NUM_COL}"); plt.legend(); plt.tight_layout(); plt.show()

        print(f"Naive forecast (next month = this month) MAE: {mae:,.2f}")
        print("Any real forecasting model has to beat that. Most do not, by much.")
    else:
        print("Fewer than 6 months of data - not enough for a trend.")
else:
    print("No date column. Note in your report that time-based analysis was not possible.")


---
## Day 4 deliverable

1. **The cross-validation table** from Step 3, with the spread column.
2. **The tuning result** — best settings, and how much they improved on your guess.
3. **Your model choice**, in the three-part sentence from Step 5.
4. **The cluster profile** from Step 6, with each cluster given a human name — or a clear statement that no real clusters exist in your data.
5. **The PCA variance number** — what % two components keep.
6. **The model card**, filled in, including at least two limits of your own.

**Bring tomorrow:** `final_model.joblib` and `model_card.txt`. Day 5 turns the model into a
recommendation, and you cannot do that without them.
